In [ ]:
from xml.etree import ElementTree
from pynq import Overlay
from stream import StreamingNPU
metadata = ElementTree.parse('board.hwh')
hp_widths = [p.get('VALUE') for p in metadata.iter('PARAMETER') if p.get('NAME') == 'PCW_S_AXI_HP0_DATA_WIDTH']
assert hp_widths and all(width == '64' for width in hp_widths), 'Use the corrected 64-bit HP0 bitstream and matching HWH'
overlay = Overlay('board.bit')
npu = StreamingNPU(overlay)
print(npu.status())
print('Streaming lanes:', npu.lanes, 'vectors per bank:', npu.max_vectors)

In [ ]:
values = [[((row * npu.lanes + lane) % 33 - 16) * 4096 for lane in range(npu.lanes)] for row in range(64)]
print(npu.submit(values, bank=0, mode='tanh', tag=1))
print(npu.submit(values, bank=1, mode='sigmoid', tag=2))

In [ ]:
import pandas as pd
from IPython.display import display
for bank in (0, 1):
    result = npu.collect(bank)
    display(pd.DataFrame({'actual': result['actual'].ravel(), 'expected': result['expected'].ravel(), 'difference': result['difference'].ravel()}))
    print('Bank', bank, 'PASS:', result['passed'], 'CPU submission cycles:', result['submit_cpu_cycles'])
    assert result['passed']

In [ ]:
for tag, mode in enumerate(('relu', 'linear'), start=3):
    print(npu.submit(values, bank=0, mode=mode, tag=tag))
    result = npu.collect(0)
    print(mode, 'PASS:', result['passed'], 'outputs checked:', result['actual'].size)
    assert result['passed']
print('All four streaming activation modes passed')

### Latency with reusable DMA buffers
Run the preceding correctness tests first. This benchmark preallocates buffers, precomputes the reference, and checks every returned batch outside its timer. The existing collect() call still checks results by default. Restart the kernel after uploading the updated stream.py and run from the first cell.


In [ ]:
import time
import statistics
import numpy as np
from client import activation_reference

mode = "tanh"
iterations = 14
benchmark_values = np.array([
    [((row * npu.lanes + lane) % 33 - 16) * 4096
     for lane in range(npu.lanes)]
    for row in range(npu.max_vectors)
], dtype=np.int32)

npu.prepare_buffers()
expected = np.array([
    [activation_reference(int(x), npu.MODES[mode], iterations) for x in row]
    for row in benchmark_values
], dtype=np.int32)

def run_timed_job(tag):
    start = time.perf_counter_ns()
    submission = npu.submit(
        benchmark_values, bank=0, mode=mode, tag=tag,
        min_iter=iterations, max_iter=iterations, full_iter=iterations
    )
    submitted = time.perf_counter_ns()
    result = npu.collect(0, check=False)
    finished = time.perf_counter_ns()

    assert result["exact_comparison"]
    np.testing.assert_array_equal(result["actual"], expected)
    return {
        "submit_us": (submitted - start) / 1000,
        "collect_us": (finished - submitted) / 1000,
        "total_us": (finished - start) / 1000,
        "submit_cpu_cycles": submission["submit_cpu_cycles"],
    }

for tag in range(5):
    run_timed_job(tag)
samples = [run_timed_job(tag) for tag in range(5, 105)]
for field in ("submit_us", "collect_us", "total_us"):
    timings = sorted(sample[field] for sample in samples)
    print(f"{field}: median={statistics.median(timings):.2f}, "
          f"min={timings[0]:.2f}, p95={timings[94]:.2f}")
median_seconds = statistics.median(s["total_us"] for s in samples) / 1e6
print(f"Effective API throughput: {expected.size / median_seconds:,.0f} outputs/second")
print(f"Correctness PASS: {len(samples)} measured batches, {expected.size} outputs each")
print("Includes Python input validation, copying, cache maintenance, DMA, "
      "CPU commands/polling, result copying, and bank release.")
print("Excludes buffer allocation and reference checking; not hardware-only latency.")


### Per-stage host timing
Run after the benchmark above. Twenty separately profiled batches identify the bottleneck. Shares use stage means divided by the combined mean; medians should not be added. Buffer lookup should not allocate after prepare_buffers(). Keep the unprofiled benchmark for headline latency.


In [ ]:
import statistics
import numpy as np

npu.prepare_buffers()
profile_samples = []
for tag in range(200, 220):
    submitted = npu.submit(
        benchmark_values, bank=0, mode=mode, tag=tag,
        min_iter=iterations, max_iter=iterations, full_iter=iterations,
        profile=True
    )
    collected = npu.collect(0, check=False, profile=True)

    assert collected["exact_comparison"]
    np.testing.assert_array_equal(collected["actual"], expected)
    profile_samples.append({"submit": submitted["timings_us"],
                            "collect": collected["timings_us"]})

mean_total = statistics.mean(
    sample["submit"]["total_us"] + sample["collect"]["total_us"]
    for sample in profile_samples
)
stages = []
for operation in ("submit", "collect"):
    for stage in profile_samples[0][operation]:
        if stage == "total_us":
            continue
        measurements = sorted(sample[operation][stage] for sample in profile_samples)
        mean = statistics.mean(measurements)
        stages.append((f"{operation}.{stage}", mean,
                       statistics.median(measurements), measurements[18],
                       100 * mean / mean_total if mean_total else 0))
print("Stage                                     mean_us   median_us      p95_us   share")
for name, mean, median, p95, share in sorted(stages, key=lambda row: row[1], reverse=True):
    print(f"{name:40s} {mean:10.2f} {median:11.2f} {p95:11.2f} {share:6.1f}%")
print(f"Profiled client-span mean: {mean_total:.2f} us")
print("Correctness PASS: 20 profiled batches; stage timers add software overhead.")
print("DMA stages include register programming/polling; mailbox includes ready checks, "
      "field writes and acknowledgement. These are host timings, not FPGA cycle counts.")
print("completion_wait is only the remaining wait when collect() starts, not total compute time.")


In [ ]:
npu.close()
print("Reusable DMA buffers released")
